https://unit8co.github.io/darts/examples/07-NBEATS-examples.html

In [1]:
import torch
import numpy as np
import pandas as pd
import shutil

from darts import TimeSeries
from darts.models import NBEATSModel
from darts.dataprocessing.transformers import Scaler, MissingValuesFiller
from darts.metrics import mape, r2_score, mae, rmse
from darts import concatenate

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import warnings

warnings.filterwarnings("ignore")
import logging

logging.disable(logging.CRITICAL)

In [2]:
def display_forecast(pred_series, ts_transformed, start_date=None):
    plt.figure(figsize=(8, 5))
    if start_date:
        ts_transformed = ts_transformed.drop_before(start_date)
    ts_transformed.univariate_component(0).plot(label="actual")
    pred_series.plot(label=("predicted"))
    plt.title(
        "R2: {}\n".format(r2_score(ts_transformed.univariate_component(0), pred_series))
        + "MAPE: {}\n".format(mape(ts_transformed.univariate_component(0), pred_series))
        + "MAE: {}\n".format(mae(ts_transformed.univariate_component(0), pred_series))
        + "RMSE: {}\n".format(rmse(ts_transformed.univariate_component(0), pred_series))
    )
    plt.legend()

def display_forecast_plotly(title_text, pred_series, ts_transformed, start_date=None):

    if start_date:
        ts_transformed = ts_transformed.drop_before(start_date)

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=ts_transformed.univariate_component(0).time_index, y=ts_transformed.univariate_component(0).pd_series(), name='actual'))
    fig.add_trace(go.Scatter(x=pred_series.time_index, y=pred_series.pd_series(), name='predicted'))

    # add title with R2, MAPE, MAE and RMSE
    fig.update_layout(title=f"{title_text}<br>R2: {r2_score(ts_transformed.univariate_component(0), pred_series)} | "
        + f"MAPE: {mape(ts_transformed.univariate_component(0), pred_series)} | "
        + f"MAE: {mae(ts_transformed.univariate_component(0), pred_series)} | "
        + f"RMSE: {rmse(ts_transformed.univariate_component(0), pred_series)}")

    fig.show()

## Get the Data & Process (merge)

In [3]:
# Load data
# df = pd.read_csv('../data/01-output-BTCUSDT_1d-from-2020-12-31 00:00:00-until-2022-12-31 00:00:00-log-return.csv')
df = pd.read_csv('../data/01-output-ETHUSDT_1d-from-2018-12-31 00:00:00-until-2020-12-31 00:00:00-log-return.csv')
# df = pd.read_csv('../data/01-output-SOLUSDT_1d-from-2020-12-31 00:00:00-until-2022-12-31 00:00:00-log-return.csv')


data_name = 'ETHUSDT_1h' # BTCUSDT_1h, ETHUSDT_1h, SOLUSDT_1h
from_date = '2019-05-31' # 2021-11-30, 2019-05-31, 2021-03-31
# until_date = '2019-05-31'

target_feature = 'processed_log_return_wtmra_0'

df_features = pd.read_csv(f'../data/02b-output-{data_name}-log-return-datetime.csv')

In [4]:
df_features

,date,processed_log_return_wtmra_0,processed_log_return_wtmra_0_year,processed_log_return_wtmra_0_month,processed_log_return_wtmra_0_day,processed_log_return_wtmra_0_hour,processed_log_return_wtmra_0_dayofweek,processed_log_return_wtmra_0_dayofyear,processed_log_return_wtmra_0_days_in_month
0,2018-12-31 00:00:00,0.168167,2018,12,31,0,0,365,31
1,2018-12-31 01:00:00,-0.097854,2018,12,31,1,0,365,31
2,2018-12-31 02:00:00,0.087226,2018,12,31,2,0,365,31
3,2018-12-31 03:00:00,-0.363237,2018,12,31,3,0,365,31
4,2018-12-31 04:00:00,0.354260,2018,12,31,4,0,365,31
...,...,...,...,...,...,...,...,...,...
26300,2021-12-30 20:00:00,-0.021361,2021,12,30,20,3,364,31
26301,2021-12-30 21:00:00,-0.175831,2021,12,30,21,3,364,31
26302,2021-12-30 22:00:00,0.283100,2021,12,30,22,3,364,31
26303,2021-12-30 23:00:00,-0.257809,2021,12,30,23,3,364,31


In [5]:
# remove processed_log_return_wtmra_0 from features
df_features.drop(columns=['processed_log_return_wtmra_0'], inplace=True)

In [6]:
# Get the features columns into a list
list_features = df_features.columns.tolist()

# Remove 'date', 'processed_log_return_wtmra_0'
list_features.remove('date')
# list_features.remove(target_feature)

print(list_features)

['processed_log_return_wtmra_0_year', 'processed_log_return_wtmra_0_month', 'processed_log_return_wtmra_0_day', 'processed_log_return_wtmra_0_hour', 'processed_log_return_wtmra_0_dayofweek', 'processed_log_return_wtmra_0_dayofyear', 'processed_log_return_wtmra_0_days_in_month']


In [7]:
# From df_features, get only the 00:00:00 rows
df_features = df_features[df_features['date'].str.contains('00:00:00')]

# Remove the 00:00:00 from the date column
df_features['date'] = df_features['date'].str.replace(' 00:00:00', '')

# inner join df (date) and df_features (end_date)
df_merged = df.merge(df_features, on='date', how='inner')

In [8]:
# Filter from_date 
df_merged = df_merged[df_merged['date'] >= from_date]
df_merged

,date,open,high,low,close,volume,original_close,processed_log_return,outliers_processed_log_return,normalized_outliers_processed_log_return,...,processed_log_return_wtmra_0_1_2_3,processed_log_return_wtmra_0_1_2_3_4,processed_log_return_wtmra_0_1_2_3_4_5,processed_log_return_wtmra_0_year,processed_log_return_wtmra_0_month,processed_log_return_wtmra_0_day,processed_log_return_wtmra_0_hour,processed_log_return_wtmra_0_dayofweek,processed_log_return_wtmra_0_dayofyear,processed_log_return_wtmra_0_days_in_month
150,2019-05-31,268.92,288.62,240.14,254.56,1.066279e+06,254.56,-0.054543,-0.054543,-0.622462,...,-0.557311,-0.672344,-0.622462,2019,5,31,0,4,151,31
151,2019-06-01,254.59,268.72,245.21,267.90,6.021539e+05,267.90,0.051077,0.051077,0.490950,...,0.563167,0.443812,0.490950,2019,6,1,0,5,152,30
152,2019-06-02,267.90,275.50,260.68,264.33,4.556770e+05,264.33,-0.013415,-0.013415,-0.188912,...,-0.112749,-0.233181,-0.188912,2019,6,2,0,6,153,30
153,2019-06-03,264.33,273.20,263.20,268.88,3.015362e+05,268.88,0.017067,0.017067,0.132423,...,0.209642,0.091048,0.132423,2019,6,3,0,0,154,30
154,2019-06-04,268.87,270.00,248.00,249.91,3.973607e+05,249.91,-0.073164,-0.073164,-0.818766,...,-0.742404,-0.857215,-0.818766,2019,6,4,0,1,155,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726,2020-12-27,626.78,652.91,615.26,637.44,9.585855e+05,637.44,0.016801,0.016801,0.129618,...,0.027948,0.119078,0.129618,2020,12,27,0,6,362,31
727,2020-12-28,637.44,717.13,625.00,685.11,1.859968e+06,685.11,0.072119,0.072119,0.712768,...,0.611838,0.705467,0.712768,2020,12,28,0,0,363,31
728,2020-12-29,685.10,748.09,681.04,730.41,1.627154e+06,730.41,0.064027,0.064027,0.627458,...,0.531260,0.623922,0.627458,2020,12,29,0,1,364,31
729,2020-12-30,730.40,740.78,689.20,732.00,1.106876e+06,732.00,0.002174,0.002174,-0.024568,...,-0.113422,-0.023807,-0.024568,2020,12,30,0,2,365,31


In [9]:
df_merged[target_feature] = df_merged[target_feature].astype('float32')

for feature in list_features:
    df_merged[feature] = df_merged[feature].astype('float32')

In [10]:
df_merged.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'original_close',
       'processed_log_return', 'outliers_processed_log_return',
       'normalized_outliers_processed_log_return',
       'processed_log_return_wtmra_0', 'processed_log_return_wtmra_1',
       'processed_log_return_wtmra_2', 'processed_log_return_wtmra_3',
       'processed_log_return_wtmra_4', 'processed_log_return_wtmra_5',
       'processed_log_return_wtmra_5_4', 'processed_log_return_wtmra_5_4_3',
       'processed_log_return_wtmra_5_4_3_2',
       'processed_log_return_wtmra_5_4_3_2_1',
       'processed_log_return_wtmra_5_4_3_2_1_0',
       'processed_log_return_wtmra_0_1', 'processed_log_return_wtmra_0_1_2',
       'processed_log_return_wtmra_0_1_2_3',
       'processed_log_return_wtmra_0_1_2_3_4',
       'processed_log_return_wtmra_0_1_2_3_4_5',
       'processed_log_return_wtmra_0_year',
       'processed_log_return_wtmra_0_month',
       'processed_log_return_wtmra_0_day', 'processed_log_return_wtmra_0

In [11]:
# Filter df_merged by a <= certain date
df_merged = df_merged[df_merged['date'] <= '2020-06-30'] # 2022-12-31, 2020-06-30, 2022-04-30

## Create the `series` object

In [12]:
# Convert to TimeSeries object
series_log_return = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=[target_feature])

series_processed_log_return_wtmra_0_year = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_year'])
series_processed_log_return_wtmra_0_month = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_month'])
series_processed_log_return_wtmra_0_day = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_day'])
series_processed_log_return_wtmra_0_hour = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_hour'])
series_processed_log_return_wtmra_0_dayofweek = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_dayofweek'])
series_processed_log_return_wtmra_0_dayofyear = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_dayofyear'])
series_processed_log_return_wtmra_0_days_in_month = TimeSeries.from_dataframe(df_merged, time_col='date', value_cols=['processed_log_return_wtmra_0_days_in_month'])


## Split the data into train, validation and test sets

In [13]:
train_date_cut = "20200430" # 20221031, 20200430, 20220228
val_date_cut = "20200531" # 20221130, 20200531, 20220331

train_log_return, temp_log_return = series_log_return.split_after(pd.Timestamp(train_date_cut))
val_log_return, test_log_return = temp_log_return.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_year, temp_processed_log_return_wtmra_0_year = series_processed_log_return_wtmra_0_year.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_year, test_processed_log_return_wtmra_0_year = temp_processed_log_return_wtmra_0_year.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_month, temp_processed_log_return_wtmra_0_month = series_processed_log_return_wtmra_0_month.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_month, test_processed_log_return_wtmra_0_month = temp_processed_log_return_wtmra_0_month.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_day, temp_processed_log_return_wtmra_0_day = series_processed_log_return_wtmra_0_day.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_day, test_processed_log_return_wtmra_0_day = temp_processed_log_return_wtmra_0_day.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_hour, temp_processed_log_return_wtmra_0_hour = series_processed_log_return_wtmra_0_hour.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_hour, test_processed_log_return_wtmra_0_hour = temp_processed_log_return_wtmra_0_hour.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_dayofweek, temp_processed_log_return_wtmra_0_dayofweek = series_processed_log_return_wtmra_0_dayofweek.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_dayofweek, test_processed_log_return_wtmra_0_dayofweek = temp_processed_log_return_wtmra_0_dayofweek.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_dayofyear, temp_processed_log_return_wtmra_0_dayofyear = series_processed_log_return_wtmra_0_dayofyear.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_dayofyear, test_processed_log_return_wtmra_0_dayofyear = temp_processed_log_return_wtmra_0_dayofyear.split_after(pd.Timestamp(val_date_cut))

train_processed_log_return_wtmra_0_days_in_month, temp_processed_log_return_wtmra_0_days_in_month = series_processed_log_return_wtmra_0_days_in_month.split_after(pd.Timestamp(train_date_cut))
val_processed_log_return_wtmra_0_days_in_month, test_processed_log_return_wtmra_0_days_in_month = temp_processed_log_return_wtmra_0_days_in_month.split_after(pd.Timestamp(val_date_cut))

In [14]:
# train_log_return.plot(label="train")
# val_log_return.plot(label="val")
# test_log_return.plot(label="test")

# Make the train, val and test sets plot using plotly go


fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=val_log_return.time_index, y=val_log_return.pd_series(), line=dict(color="red"), name='val'))
fig.add_trace(go.Scatter(x=test_log_return.time_index, y=test_log_return.pd_series(), line=dict(color="red"), name='val'))

fig.update_layout(title=f'Train, val and test sets for feature: {target_feature}')

fig.show()

## Scale the data

In [15]:
# Normalize
scaler_processed_log_return_wtmra_0_year = Scaler()
scaler_processed_log_return_wtmra_0_month = Scaler()
scaler_processed_log_return_wtmra_0_day = Scaler()
scaler_processed_log_return_wtmra_0_hour = Scaler()
scaler_processed_log_return_wtmra_0_dayofweek = Scaler()
scaler_processed_log_return_wtmra_0_dayofyear = Scaler()
scaler_processed_log_return_wtmra_0_days_in_month = Scaler()

train_processed_log_return_wtmra_0_year_scaled = scaler_processed_log_return_wtmra_0_year.fit_transform(train_processed_log_return_wtmra_0_year)
val_processed_log_return_wtmra_0_year_scaled = scaler_processed_log_return_wtmra_0_year.transform(val_processed_log_return_wtmra_0_year)
test_processed_log_return_wtmra_0_year_scaled = scaler_processed_log_return_wtmra_0_year.transform(test_processed_log_return_wtmra_0_year)

train_processed_log_return_wtmra_0_month_scaled = scaler_processed_log_return_wtmra_0_month.fit_transform(train_processed_log_return_wtmra_0_month)
val_processed_log_return_wtmra_0_month_scaled = scaler_processed_log_return_wtmra_0_month.transform(val_processed_log_return_wtmra_0_month)
test_processed_log_return_wtmra_0_month_scaled = scaler_processed_log_return_wtmra_0_month.transform(test_processed_log_return_wtmra_0_month)

train_processed_log_return_wtmra_0_day_scaled = scaler_processed_log_return_wtmra_0_day.fit_transform(train_processed_log_return_wtmra_0_day)
val_processed_log_return_wtmra_0_day_scaled = scaler_processed_log_return_wtmra_0_day.transform(val_processed_log_return_wtmra_0_day)
test_processed_log_return_wtmra_0_day_scaled = scaler_processed_log_return_wtmra_0_day.transform(test_processed_log_return_wtmra_0_day)

train_processed_log_return_wtmra_0_hour_scaled = scaler_processed_log_return_wtmra_0_hour.fit_transform(train_processed_log_return_wtmra_0_hour)
val_processed_log_return_wtmra_0_hour_scaled = scaler_processed_log_return_wtmra_0_hour.transform(val_processed_log_return_wtmra_0_hour)
test_processed_log_return_wtmra_0_hour_scaled = scaler_processed_log_return_wtmra_0_hour.transform(test_processed_log_return_wtmra_0_hour)

train_processed_log_return_wtmra_0_dayofweek_scaled = scaler_processed_log_return_wtmra_0_dayofweek.fit_transform(train_processed_log_return_wtmra_0_dayofweek)
val_processed_log_return_wtmra_0_dayofweek_scaled = scaler_processed_log_return_wtmra_0_dayofweek.transform(val_processed_log_return_wtmra_0_dayofweek)
test_processed_log_return_wtmra_0_dayofweek_scaled = scaler_processed_log_return_wtmra_0_dayofweek.transform(test_processed_log_return_wtmra_0_dayofweek)

train_processed_log_return_wtmra_0_dayofyear_scaled = scaler_processed_log_return_wtmra_0_dayofyear.fit_transform(train_processed_log_return_wtmra_0_dayofyear)
val_processed_log_return_wtmra_0_dayofyear_scaled = scaler_processed_log_return_wtmra_0_dayofyear.transform(val_processed_log_return_wtmra_0_dayofyear)
test_processed_log_return_wtmra_0_dayofyear_scaled = scaler_processed_log_return_wtmra_0_dayofyear.transform(test_processed_log_return_wtmra_0_dayofyear)

train_processed_log_return_wtmra_0_days_in_month_scaled = scaler_processed_log_return_wtmra_0_days_in_month.fit_transform(train_processed_log_return_wtmra_0_days_in_month)
val_processed_log_return_wtmra_0_days_in_month_scaled = scaler_processed_log_return_wtmra_0_days_in_month.transform(val_processed_log_return_wtmra_0_days_in_month)
test_processed_log_return_wtmra_0_days_in_month_scaled = scaler_processed_log_return_wtmra_0_days_in_month.transform(test_processed_log_return_wtmra_0_days_in_month)

## Create `past_covariates`

In [16]:
train_past_covariates = concatenate([train_processed_log_return_wtmra_0_year_scaled, train_processed_log_return_wtmra_0_month_scaled, train_processed_log_return_wtmra_0_day_scaled, train_processed_log_return_wtmra_0_hour_scaled, train_processed_log_return_wtmra_0_dayofweek_scaled, train_processed_log_return_wtmra_0_dayofyear_scaled, train_processed_log_return_wtmra_0_days_in_month_scaled], axis=1)
val_past_covariates = concatenate([val_processed_log_return_wtmra_0_year_scaled, val_processed_log_return_wtmra_0_month_scaled, val_processed_log_return_wtmra_0_day_scaled, val_processed_log_return_wtmra_0_hour_scaled, val_processed_log_return_wtmra_0_dayofweek_scaled, val_processed_log_return_wtmra_0_dayofyear_scaled, val_processed_log_return_wtmra_0_days_in_month_scaled], axis=1)
test_past_covariates = concatenate([test_processed_log_return_wtmra_0_year_scaled, test_processed_log_return_wtmra_0_month_scaled, test_processed_log_return_wtmra_0_day_scaled, test_processed_log_return_wtmra_0_hour_scaled, test_processed_log_return_wtmra_0_dayofweek_scaled, test_processed_log_return_wtmra_0_dayofyear_scaled, test_processed_log_return_wtmra_0_days_in_month_scaled], axis=1)
past_covariates = concatenate([train_past_covariates, val_past_covariates, test_past_covariates], axis=0)

## Training and validade Univariate model

In [25]:
window_size = 7 
num_forecast_points = 1
df_metrics = pd.DataFrame()

In [18]:
uni_model_nbeats = NBEATSModel(
    input_chunk_length=window_size,
    output_chunk_length=num_forecast_points,
    batch_size= 2 * (window_size + num_forecast_points),
    random_state=0,
    n_epochs=100,
    num_layers=2,
    layer_widths=512,
    loss_fn=torch.nn.MSELoss(),
)

In [19]:
uni_model_nbeats.fit(train_log_return,
                    val_series=val_log_return, 
                    verbose=False)

NBEATSModel(output_chunk_shift=0, generic_architecture=True, num_stacks=30, num_blocks=1, num_layers=2, layer_widths=512, expansion_coefficient_dim=5, trend_polynomial_degree=2, dropout=0.0, activation=ReLU, input_chunk_length=7, output_chunk_length=1, batch_size=16, random_state=0, n_epochs=100, loss_fn=MSELoss())

In [20]:
# concatenate test and val series
test_val_log_return = concatenate([val_log_return, test_log_return], axis=0)

In [21]:
pred = uni_model_nbeats.predict(n=(len(val_log_return) + len(test_log_return)))

fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=test_val_log_return.time_index, y=test_val_log_return.pd_series(), name='test'))
fig.add_trace(go.Scatter(x=pred.time_index, y=pred.pd_series(), name='forecast'))

# add title with R2, MAPE, MAE and RMSE
fig.update_layout(title="Univariate Results<br>Evaluation Metrics: R2: {}\n".format(r2_score(series_log_return, pred))
    + " MAPE: {}".format(mape(series_log_return, pred))
    + " MAE: {}".format(mae(series_log_return, pred))
    + " RMSE: {}".format(rmse(series_log_return, pred)))

fig.show()

# series_log_return.plot(label="actual")
# pred.plot(label="forecast")
# plt.legend()
# print("MAPE = {:.2f}%".format(mape(series_log_return, pred)))
# print("MAE = {:.5f}".format(mae(series_log_return, pred)))
# print("RMSE = {:.5f}".format(rmse(series_log_return, pred)))
# print("R2 = {:.2f}".format(r2_score(series_log_return, pred)))

Predicting: |          | 0/? [00:00<?, ?it/s]

In [22]:
# uni_pred_series_log_return = uni_model_nbeats.historical_forecasts(
#     test_log_return,
#     forecast_horizon=num_forecast_points,
#     stride=1,
#     retrain=False,
#     verbose=False,
# )

In [23]:
# display_forecast_plotly("Univariate", uni_pred_series_log_return, test_log_return)

In [26]:
# Export a .json with the CCY, scenario, R2, MAPE, MAE and RMSE
dict_metrics = {
    "CCY": data_name,
    "scenario": "datetime",
    "type": "univariate",
    "R2": r2_score(series_log_return, pred),
    "MAPE": mape(series_log_return, pred),
    "MAE": mae(series_log_return, pred),
    "RMSE": rmse(series_log_return, pred)
}

# Append a dict to the df_metrics
df_metrics = df_metrics._append(dict_metrics, ignore_index=True)

## Training and validade Multivariate model

In [27]:
multi_model_nbeats = NBEATSModel(
    input_chunk_length=window_size,
    output_chunk_length=num_forecast_points,
    batch_size= 2 * (window_size + num_forecast_points),
    random_state=0,
    n_epochs=100,
    num_layers=2,
    layer_widths=512,
    loss_fn=torch.nn.MSELoss(),
)

In [28]:
# fit using multiple (two) target series
multi_model_nbeats.fit(train_log_return,
          val_series=val_log_return,
          past_covariates=train_past_covariates,
          val_past_covariates=val_past_covariates,
          verbose=False
          )

NBEATSModel(output_chunk_shift=0, generic_architecture=True, num_stacks=30, num_blocks=1, num_layers=2, layer_widths=512, expansion_coefficient_dim=5, trend_polynomial_degree=2, dropout=0.0, activation=ReLU, input_chunk_length=7, output_chunk_length=1, batch_size=16, random_state=0, n_epochs=100, loss_fn=MSELoss())

In [29]:
pred = multi_model_nbeats.predict(n=(len(val_log_return) + len(test_log_return)), series=train_log_return, past_covariates=past_covariates)

fig = go.Figure()

fig.add_trace(go.Scatter(x=train_log_return.time_index, y=train_log_return.pd_series(), name='train'))
fig.add_trace(go.Scatter(x=test_val_log_return.time_index, y=test_val_log_return.pd_series(), name='test'))
fig.add_trace(go.Scatter(x=pred.time_index, y=pred.pd_series(), name='forecast'))

# add title with R2, MAPE, MAE and RMSE
fig.update_layout(title="Multivariate Results<br>Evaluation Metrics: R2: {}\n".format(r2_score(series_log_return, pred))
    + " MAPE: {}".format(mape(series_log_return, pred))
    + " MAE: {}".format(mae(series_log_return, pred))
    + " RMSE: {}".format(rmse(series_log_return, pred)))

fig.show()

# series_log_return.plot(label="actual")
# pred.plot(label="forecast")
# plt.legend()
# print("MAPE = {:.2f}%".format(mape(series_log_return, pred)))
# print("MAE = {:.5f}".format(mae(series_log_return, pred)))
# print("RMSE = {:.5f}".format(rmse(series_log_return, pred)))
# print("R2 = {:.2f}".format(r2_score(series_log_return, pred)))

Predicting: |          | 0/? [00:00<?, ?it/s]

In [30]:
# multi_pred_series_log_return = multi_model_nbeats.historical_forecasts(
#     test_log_return,
#     past_covariates=test_past_covariates,
#     forecast_horizon=num_forecast_points,
#     stride=1,
#     retrain=False,
#     verbose=False,
# )

In [31]:
# display_forecast_plotly("Multivariate", multi_pred_series_log_return, test_log_return)

In [32]:
# Export a .json with the CCY, scenario, R2, MAPE, MAE and RMSE
dict_metrics = {
    "CCY": data_name,
    "scenario": "datetime",
    "type": "multivariate",
    "R2": r2_score(series_log_return, pred),
    "MAPE": mape(series_log_return, pred),
    "MAE": mae(series_log_return, pred),
    "RMSE": rmse(series_log_return, pred)
}

# Append a dict to the df_metrics
df_metrics = df_metrics._append(dict_metrics, ignore_index=True)

In [33]:
df_metrics.to_csv(f'outputs/04-02b-output-metrics.csv', index=False)